<a href="https://colab.research.google.com/github/hashirama21/gik-icechain/blob/main/notebooks/gik_icechain_walkthrough.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GIK-IceChain - Live Walkthrough (MinIO, real execution)

Runs the pipeline **component by component on real data from the live MinIO store**
(no synthetic fallback), then the full pipeline, then live benchmarks:

1. **Setup** - locate repo, read MinIO credentials, download public prerequisites.
2. **C1** - open the real IceChunk store; read a real forecast day's `tp`.
3. **C2** - compute real exceedance on that forecast, and read the live exceedance Zarr.
4. **C3** - real CRMA risk for admin-1 units from the live exceedance store.
5. **Full pipeline** - `gik-icechain run-all` against MinIO.
6. **Benchmarks** - live `run_benchmark` against the store.

Requires MinIO credentials (`MINIO_ENDPOINT_URL` / `MINIO_ACCESS_KEY` /
`MINIO_SECRET_KEY`, via env, Colab secrets, or `.env`). It does **not** fall back to
synthetic data - if the store is unreachable, the cells raise.

## 0. Setup - repo, MinIO credentials, prerequisites

In [1]:
import os, sys, subprocess
from pathlib import Path

def _bootstrap() -> Path:
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p / "configs" / "default.yaml").exists():
            return p
    repo = Path.cwd() / "gik-icechain"
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/hashirama21/gik-icechain.git", str(repo)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"{repo}[dev]"], check=True)

    # Fix for missing system-level config directory
    sys_config_dir = Path("/usr/local/lib/python3.12/configs")
    if not sys_config_dir.exists():
        subprocess.run(["sudo", "mkdir", "-p", str(sys_config_dir)], check=True)
        subprocess.run(["sudo", "ln", "-sf", str(repo / "configs" / "default.yaml"), str(sys_config_dir / "default.yaml")], check=True)

    return repo

REPO = _bootstrap()
DATA = REPO / "data"
sys.path.insert(0, str(REPO / "src"))

def _secret(name: str, default: str = "") -> str:
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v: return v
    except Exception: pass
    return os.environ.get(name, default)

# Env setup
_minio = _secret("MINIO") or _secret("MINIO_ENDPOINT_URL")
ENDPOINT = _minio if _minio.startswith("http") else (f"http://{_minio}" if _minio else "")
KEY = _secret("MINIO_ACCESS_KEY") or _secret("AWS_ACCESS_KEY_ID")
SECRET = _secret("MINIO_SECRET_KEY") or _secret("AWS_SECRET_ACCESS_KEY")

os.environ["AWS_ENDPOINT_URL"] = ENDPOINT
os.environ["AWS_ACCESS_KEY_ID"] = KEY
os.environ["AWS_SECRET_ACCESS_KEY"] = SECRET
os.environ.setdefault("AWS_REGION", "eu-west-1")
os.environ.setdefault("ECCODES_PYTHON_USE_FINDLIBS", "1")

# Ensure data directory and thresholds exist
DATA.mkdir(exist_ok=True)
tools_script = str(REPO / "scripts" / "tools.py")
# Force download of thresholds to the expected location
subprocess.run([sys.executable, tools_script, "download-thresholds"], cwd=str(REPO), check=True)

from gik_icechain.shared.config import load_config
cfg = load_config(REPO / CONFIG)
print("Repo:", REPO)
print("Thresholds check:", "OK" if (DATA / "cmorph_thresholds").exists() else "MISSING")

RuntimeError: MinIO credentials required (MINIO_ENDPOINT_URL/MINIO_ACCESS_KEY/MINIO_SECRET_KEY). This notebook runs against the live store - no synthetic fallback.

## 1. Component 1 - real IceChunk store

Open the live store, list its committed days, and read one real forecast day's `tp`
(subset to the bbox so the GRIB decode is fast). No synthetic data.

In [ ]:
from gik_icechain.conversion.icechunk_writer import IceChainStore

store = IceChainStore(cfg.outputs.icechunk_store_uri,
                      region=cfg.outputs.icechunk_store_region, endpoint_url=ENDPOINT)
store.create_or_open()
report = store.validate()
print(f"Committed days: {report['committed_days']} | range {report['date_range']}")
snaps = store.list_snapshots()  # already sorted by forecast_date
DAY = snaps[-1]["forecast_date"]                       # latest real forecast day in the store
print(f"Reading forecast day: {DAY}")

session = store.readonly_session()
# Subset members/steps so the live GRIB decode stays minutes-scale (real ECMWF
# data, just fewer chunks); the full pipeline in section 4 uses all 51 members.
day_ds = (xr.open_zarr(session.store, group=DAY, consolidated=False)[["tp"]]
          .sel(**BBOX).isel(member=slice(0, 6), step=slice(0, 41)))
tp_mm = (day_ds["tp"] * cfg.component2.precip_scale_to_mm).load()   # m -> mm; triggers real decode
print(f"tp (bbox) shape : {tuple(tp_mm.shape)}  (member, step, lat, lon)")
print(f"members={day_ds.sizes.get('member')} steps={day_ds.sizes.get('step')} "
      f"| tp max={float(tp_mm.max()):.1f} mm")


## 2. Component 2 - real exceedance

Compute real rolling accumulations + adaptive GEV exceedance on the decoded forecast,
then read back the **live exceedance Zarr** (the persisted C2 output) from MinIO.

In [ ]:
from gik_icechain.exceedance.accumulations import compute_rolling_accumulations
from gik_icechain.exceedance.thresholds import (
    AdaptiveGEVThresholds, ClimateMode, classify_enso, classify_iod, get_season,
)
from gik_icechain.exceedance.exceedance import compute_exceedance_probabilities

acc = compute_rolling_accumulations(xr.Dataset({"tp": tp_mm}), windows_h=[24, 72])
thresholds = AdaptiveGEVThresholds.load(DATA / "cmorph_thresholds")
enso_iod = (pd.read_csv(DATA / "enso_iod_index.csv", parse_dates=["date"])
            .set_index("date").sort_index())
d0 = pd.Timestamp(DAY)
row = enso_iod.loc[enso_iod.index.asof(d0)]
mode = ClimateMode(get_season(d0.month),
                   classify_enso(float(row["nino34_anom"])), classify_iod(float(row["dmi"])))
thr24 = thresholds.get(24, 5, mode)
p = compute_exceedance_probabilities(acc, xr.Dataset({"rp_5y": thr24}), 24, 5, "member")
print(f"Climate mode {mode.key} | computed exceedance 24h/5yr: "
      f"max={float(p.max()):.3f} mean={float(p.mean()):.3f}")

# Live C2 output already persisted in MinIO:
exc_ds = xr.open_zarr(cfg.outputs.exceedance_store_uri, consolidated=False,
                      storage_options=STORAGE_OPTIONS)
dates = [str(x)[:10] for x in np.sort(exc_ds["date"].values)]
print(f"Live exceedance store: {len(dates)} dates {dates[0]}..{dates[-1]} | "
      f"windows {list(exc_ds['window'].values)} | RPs {list(exc_ds['return_period'].values)}")


## 3. Component 3 - real CRMA risk

Build the BN and infer admin-1 risk from the **live exceedance store** for one of its
real dates (max-aggregated to each unit's bbox).

In [ ]:
import geopandas as gpd
from gik_icechain.risk.crma_model import CRMAModel, CRMAEvidence

model = CRMAModel(crma_cfg=cfg.component3.crma_model)
model.build()
admin = gpd.read_file(DATA / "admin_boundaries" / "east_africa_admin1.geojson")

target = pd.Timestamp(dates[-1])
exc_day = exc_ds.sel(date=target, method="nearest")
lat_dim, lon_dim = cfg.component2.spatial.lat_dim, cfg.component2.spatial.lon_dim
rows = []
for _, unit in admin.iterrows():
    b = unit.geometry.bounds  # (minx, miny, maxx, maxy)
    sub = exc_day["exceedance_prob"].sel({lat_dim: slice(b[1], b[3]), lon_dim: slice(b[0], b[2])})
    if sub.sizes.get(lat_dim, 0) == 0 or sub.sizes.get(lon_dim, 0) == 0:
        continue
    p24 = float(sub.sel(window=24, return_period=5).max())
    p72 = float(sub.sel(window=72, return_period=5).max())
    ev = CRMAEvidence(exceedance_prob_24h=p24, exceedance_prob_72h=p72, exceedance_prob_7d=p72,
                      gpm_obs_24h=0.0, api_mm=20.0,
                      spatial_coverage_fraction=float((sub.sel(window=24, return_period=5) > 0.15).mean()),
                      consecutive_signal_days=1, sat_consecutive_days=0)
    rows.append({"pcode": unit.get("admin1_pcode"), "risk": model.infer(ev)["risk_label"]})
risk_df = pd.DataFrame(rows)
print(f"C3 risk for {str(target)[:10]} - {len(risk_df)} units:")
print(risk_df["risk"].value_counts().to_string())


In [ ]:
colors = {"Green": "#2ecc71", "Yellow": "#f1c40f", "Orange": "#e67e22", "Red": "#e74c3c"}
counts = risk_df["risk"].value_counts().reindex(["Green", "Yellow", "Orange", "Red"], fill_value=0)
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(counts.index, counts.values, color=[colors[s] for s in counts.index])
ax.set_title(f"Live CRMA risk distribution - {str(target)[:10]}"); ax.set_ylabel("admin-1 units")
fig.tight_layout(); plt.show()


## 4. Full pipeline - `gik-icechain run-all` against MinIO

Runs C1 → C2 → C3 for one real day and writes `*_risk_scores.json`. This re-reads
ECMWF S3 + writes the MinIO stores, so it takes several minutes.

In [ ]:
OUT = REPO / "results" / "nb_live"
cmd = [sys.executable, "-m", "gik_icechain", "run-all",
       "--start", DAY, "--end", DAY,
       "--config", str(REPO / "configs" / "default.yaml"), "--output", str(OUT)]
print("Running:", " ".join(cmd))
rc = subprocess.run(cmd, env=os.environ).returncode
print("Pipeline OK" if rc == 0 else f"Pipeline failed (exit {rc})")

import glob, json
files = sorted(glob.glob(str(OUT / "admin1_risk" / "*_risk_scores.json")))
for f in files:
    doc = json.load(open(f))
    c = {}
    for u in doc["units"].values():
        c[u["risk_label"]] = c.get(u["risk_label"], 0) + 1
    print(f"{doc['date']}: {len(doc['units'])} units -> " +
          ", ".join(f"{k}: {v}" for k, v in sorted(c.items())))


## 5. Validation metrics (EM-DAT)

Reproduce the EM-DAT validation on the April-2024 compound-flood window. C3 risk-only
reads the **live exceedance Zarr** (fast - no GRIB decode), then `validate-emdat`
reports unit-day AUC + **event-level early-detection** recall (the operational metric).

In [ ]:
VAL_START, VAL_END = "2024-04-22", "2024-04-28"   # in the live exceedance store
VAL_OUT = REPO / "results" / "admin1_risk"
tools = [sys.executable, str(REPO / "scripts" / "tools.py")]
# CHIRPS observed rainfall for the C3 API node (best-effort).
subprocess.run([*tools, "download-gpm", "--source", "chirps",
                "--start", "2024-04-08", "--end", VAL_END], env=os.environ, check=False)
# C3 risk over the flood window from the live exceedance store (MinIO via AWS_ENDPOINT_URL).
subprocess.run([sys.executable, "-m", "gik_icechain", "risk",
                "--exceedance-store", cfg.outputs.exceedance_store_uri,
                "--output", str(VAL_OUT), "--start", VAL_START, "--end", VAL_END,
                "--config", str(REPO / "configs" / "default.yaml")], env=os.environ, check=True)
# EM-DAT validation (unit-day + event-level early detection).
subprocess.run([*tools, "validate-emdat", "--risk-dir", str(VAL_OUT),
                "--start", VAL_START, "--end", VAL_END,
                "--event-level", "--lead-days", "1"], env=os.environ, check=False)

## 6. Benchmarks - live store

Measures time-to-first-byte and full-scan against the live IceChunk store and writes a
CSV to `results/benchmarks/`.

In [ ]:
from gik_icechain.conversion.benchmark import run_benchmark

RESULTS_DIR = REPO / "results" / "benchmarks"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
results = run_benchmark(gik_store_uri=cfg.outputs.icechunk_store_uri, domain="east_africa",
                        n_days=3, n_workers=4, output_dir=str(RESULTS_DIR))
for name, r in results.items():
    print(f"{name}: TTFB={r.time_to_first_byte_s:.2f}s  scan={r.full_scan_elapsed_s:.1f}s  "
          f"store={r.store_size_gb:.1f} GB")
print("\nLive walkthrough complete: C1 -> C2 -> C3 -> full pipeline -> benchmarks (all on MinIO).")
